2. CHART SETUP REQUIREMENTS
Moving Averages (Required on All Timeframes)

Indicator Period Purpose

9 EMA 9 Fast momentum indicator – triggers and short-term direction

18 EMA 18 Medium momentum – confirms trend establishment

50 EMA 50 BASELINE – Primary reference for all entries (the 'touch')

50 SMA 50 Secondary baseline – used for M3 Equilibrium entries

The 50 EMA is referred to as the baseline throughout this manual. This is the single most important level in the

Madstrat methodology. Every valid entry requires price to touch or interact with the baseline.

In [32]:
# import smartmoneyconcepts indicator
# from smartmoneyconcepts import smc
# import pandas for data manipulation
import pandas as pd
import numpy as np
# import pandas_ta as ta //require python 3.12
import MetaTrader5 as mt5
from datetime import datetime
# import pytz module for working with time zone
import pytz

In [2]:
from pathlib import Path
import sys
sys.path.append(str(Path(__file__).resolve().parents[1]))   # adds .../src/
RAW_DATA_DIR   = Path(__file__).resolve().parents[1]
RAW_DATA_DIR

NameError: name '__file__' is not defined

In [2]:
# write a function to calculate ema
def ema(df, period):
    return df['close'].ewm(span=period, adjust=False).mean()

In [3]:
# load csv file into a pandas dataframe
df = pd.read_csv('XAUUSD_5m.csv')
# convert the 'time' column to datetime format
df['time'] = pd.to_datetime(df['time'], unit='s')
df

,time,open,high,low,close,Volume
0,2025-09-21 22:00:00,3684.78,3687.91,3683.81,3687.54,1075
1,2025-09-21 22:05:00,3687.69,3687.69,3683.66,3687.18,1146
2,2025-09-21 22:10:00,3687.16,3688.01,3685.51,3687.92,1129
3,2025-09-21 22:15:00,3688.07,3688.54,3685.95,3686.86,967
4,2025-09-21 22:20:00,3686.82,3691.07,3686.75,3691.05,913
...,...,...,...,...,...,...
40603,2026-04-20 18:30:00,4813.43,4815.69,4812.00,4814.96,1859
40604,2026-04-20 18:35:00,4815.03,4815.47,4810.98,4812.07,1723
40605,2026-04-20 18:40:00,4812.10,4815.38,4812.10,4814.42,1484
40606,2026-04-20 18:45:00,4814.41,4816.13,4813.96,4815.06,1761


In [13]:
# add ema9, 18 and 50 to the dataframe AND add 50 period sma to the dataframe and made a function
def add_indicators(df):
    df['ema9'] = ema(df, 9)
    df['ema18'] = ema(df, 18)
    df['ema50'] = ema(df, 50)
    # add 50 period sma to the dataframe
    df['sma50'] = df['close'].rolling(50).mean()
    return df
df = add_indicators(df)
df

,time,open,high,low,close,Volume,ema9,ema18,ema50,sma50
0,2025-09-21 22:00:00,3684.78,3687.91,3683.81,3687.54,1075,3687.540000,3687.540000,3687.540000,NaN
1,2025-09-21 22:05:00,3687.69,3687.69,3683.66,3687.18,1146,3687.468000,3687.502105,3687.525882,NaN
2,2025-09-21 22:10:00,3687.16,3688.01,3685.51,3687.92,1129,3687.558400,3687.546094,3687.541338,NaN
3,2025-09-21 22:15:00,3688.07,3688.54,3685.95,3686.86,967,3687.418720,3687.473874,3687.514619,NaN
4,2025-09-21 22:20:00,3686.82,3691.07,3686.75,3691.05,913,3688.144976,3687.850308,3687.653261,NaN
...,...,...,...,...,...,...,...,...,...,...
40603,2026-04-20 18:30:00,4813.43,4815.69,4812.00,4814.96,1859,4812.709439,4810.634340,4807.768313,4806.1456
40604,2026-04-20 18:35:00,4815.03,4815.47,4810.98,4812.07,1723,4812.581551,4810.785462,4807.937007,4805.9146
40605,2026-04-20 18:40:00,4812.10,4815.38,4812.10,4814.42,1484,4812.949241,4811.168045,4808.191242,4805.8338
40606,2026-04-20 18:45:00,4814.41,4816.13,4813.96,4815.06,1761,4813.371393,4811.577724,4808.460605,4805.8524


In [28]:
# resample the df to 15 minutes, 1 hour and 4 hour timeframes, daily and weekly
df_15m = df.resample('15T', on='time').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'})
df_1h = df.resample('1H', on='time').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'})
df_4h = df.resample('4H', on='time').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'})
# df_daily = df.resample('D', on='time').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'})
df_weekly = df.resample('W', on='time').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'})

In [21]:
df_daily = df.resample('D', on='time').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'})
df_daily.dropna(inplace=True)
df_daily['pdh'] = df_daily['high'].shift(1)
df_daily['pdl'] = df_daily['low'].shift(1)

In [31]:
df_15m["pdh"] = df_daily["pdh"]
df_15m["pdl"] = df_daily["pdl"]
df_15m.ffill(inplace=True)
df_15m.dropna(inplace=True)
df_15m

,open,high,low,close,pdh,pdl
time,,,,,,
2025-09-22 00:00:00,3686.01,3690.75,3685.76,3690.58,3692.11,3683.66
2025-09-22 00:15:00,3690.57,3690.84,3687.70,3689.58,3692.11,3683.66
2025-09-22 00:30:00,3689.59,3690.73,3688.88,3689.53,3692.11,3683.66
2025-09-22 00:45:00,3689.53,3691.07,3687.49,3690.94,3692.11,3683.66
2025-09-22 01:00:00,3690.92,3695.36,3686.78,3694.61,3692.11,3683.66
...,...,...,...,...,...,...
2026-04-20 17:45:00,4809.39,4809.84,4805.64,4806.69,4794.95,4737.07
2026-04-20 18:00:00,4806.82,4815.97,4806.82,4815.35,4794.95,4737.07
2026-04-20 18:15:00,4815.36,4815.62,4811.93,4813.44,4794.95,4737.07


In [23]:
df_ = df_15m.join(df_daily[['pdh', 'pdl']]) 
df_daily

,open,high,low,close,pdh,pdl
time,,,,,,
2025-09-21,3684.78,3692.11,3683.66,3686.23,NaN,NaN
2025-09-22,3686.01,3749.19,3685.51,3747.87,3692.11,3683.66
2025-09-23,3748.06,3791.14,3736.45,3763.84,3749.19,3685.51
2025-09-24,3763.82,3779.23,3717.39,3742.92,3791.14,3736.45
2025-09-25,3742.87,3761.61,3722.09,3744.72,3779.23,3717.39
...,...,...,...,...,...,...
2026-04-15,4837.13,4871.31,4786.19,4823.54,4850.71,4751.46
2026-04-16,4823.50,4838.31,4772.95,4791.98,4871.31,4786.19
2026-04-17,4791.50,4889.44,4767.52,4834.06,4838.31,4772.95


In [ ]:
# make a function that takes a df, resample it to daily and calculates pdh and pdl and adds them to the df
def add_pdh_pdl(df):
    df_daily = df.resample('D', on='time').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last'})
    df_daily['pdh'] = df_daily['high'].shift(1)
    df_daily['pdl'] = df_daily['low'].shift(1)
    # merge the daily df with the original df on the time column
    df = pd.merge(df, df_daily[['pdh', 'pdl']], left_on=df['time'].dt.date, right_index=True, how='left')
    return df

df = add_pdh_pdl(df)
df


KeyError: 'The grouper name time is not found'

In [9]:
df_15m = add_indicators(df_15m)
df_15m

,open,high,low,close,ema9,ema18,ema50,sma50
time,,,,,,,,
2025-09-21 22:00:00,3684.78,3688.01,3683.66,3687.92,3687.920000,3687.920000,3687.920000,NaN
2025-09-21 22:15:00,3688.07,3692.11,3685.95,3691.26,3688.588000,3688.271579,3688.050980,NaN
2025-09-21 22:30:00,3691.22,3691.26,3689.04,3690.63,3688.996400,3688.519834,3688.152118,NaN
2025-09-21 22:45:00,3690.72,3691.13,3687.53,3688.40,3688.877120,3688.507220,3688.161839,NaN
2025-09-21 23:00:00,3688.42,3689.26,3686.39,3686.99,3688.499696,3688.347512,3688.115885,NaN
...,...,...,...,...,...,...,...,...
2026-04-20 17:45:00,4809.39,4809.84,4805.64,4806.69,4806.679182,4806.270814,4802.000039,4800.8164
2026-04-20 18:00:00,4806.82,4815.97,4806.82,4815.35,4808.413346,4807.226518,4802.523567,4801.5054
2026-04-20 18:15:00,4815.36,4815.62,4811.93,4813.44,4809.418676,4807.880568,4802.951662,4801.9544


In [ ]:
# establish connection to MetaTrader 5 terminal
if not mt5.initialize():
    print("initialize() failed, error code =",mt5.last_error())
    quit()
 
# set time zone to UTC
timezone = pytz.timezone("Etc/UTC")
# create 'datetime' object in UTC time zone to avoid the implementation of a local time zone offset
# utc_from = datetime(2020, 1, 10, tzinfo=timezone)
# get 10 EURUSD H4 bars starting from 01.10.2020 in UTC time zone
rates = mt5.copy_rates_from_pos("XAUUSD", mt5.TIMEFRAME_H4, 0, 1000)
df1 = pd.DataFrame(rates)
# df = df.set_index('time')
df1.time = pd.to_datetime(df1.time, unit='s')


initialize() failed, error code = (-6, 'Terminal: Authorization failed')


KeyboardInterrupt: 

: 

In [5]:
# Identify current trend using last candles HighLow column value, if it is 1 then the trend is up, if it is -1 then the trend is down, if it is 0 then the trend is sideways
trend_4h = df1['HighLow'].iloc[-1]
print(f"current 4h trend is {trend_4h}") #trend_4h
df1

current 4h trend is -1.0


,time,open,high,low,close,tick_volume,spread,real_volume,HighLow,level
0,2025-08-05 08:00:00,3371.02,3375.46,3365.70,3366.01,14304,21,0,-1.0,3365.70
1,2025-08-05 12:00:00,3366.02,3366.91,3349.78,3366.08,16600,21,0,NaN,NaN
2,2025-08-05 16:00:00,3366.18,3390.34,3365.83,3382.04,25905,21,0,NaN,NaN
3,2025-08-05 20:00:00,3382.05,3382.75,3377.96,3378.86,10833,21,0,NaN,NaN
4,2025-08-06 00:00:00,3381.14,3385.20,3378.06,3383.43,5865,21,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
995,2026-03-27 04:00:00,4377.79,4449.23,4376.93,4445.67,33527,61,0,NaN,NaN
996,2026-03-27 08:00:00,4445.69,4474.74,4423.42,4423.87,38678,58,0,NaN,NaN
997,2026-03-27 12:00:00,4423.82,4444.93,4404.04,4437.58,41672,58,0,NaN,NaN
998,2026-03-27 16:00:00,4437.67,4555.14,4413.55,4515.94,63327,58,0,NaN,NaN


In [16]:
rates2 = mt5.copy_rates_from_pos("XAUUSD", mt5.TIMEFRAME_H1, 0, 1000)
df2 = pd.DataFrame(rates2)
# df = df.set_index('time')
df2.time = pd.to_datetime(df2.time, unit='s')
swing_highs_lows2 = smc.swing_highs_lows(df2, swing_length = 50)
# change the name of column tick_volume to volume
df2.rename(columns={'tick_volume': 'volume'}, inplace=True)
# add swing highs and lows to the original dataframe
df2['HighLow'] = swing_highs_lows2['HighLow']
df2['level'] = swing_highs_lows2['Level']
df2

,time,open,high,low,close,volume,spread,real_volume,HighLow,level
0,2026-01-27 11:00:00,5079.35,5099.46,5077.69,5098.71,10144,27,0,-1.0,5077.69
1,2026-01-27 12:00:00,5098.45,5098.45,5081.60,5086.24,9529,27,0,NaN,NaN
2,2026-01-27 13:00:00,5086.25,5090.72,5067.01,5076.74,9246,27,0,NaN,NaN
3,2026-01-27 14:00:00,5076.79,5090.97,5073.46,5084.95,11599,27,0,NaN,NaN
4,2026-01-27 15:00:00,5084.86,5092.76,5069.61,5077.33,14447,27,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
995,2026-03-27 19:00:00,4523.82,4533.54,4505.36,4515.94,13844,58,0,NaN,NaN
996,2026-03-27 20:00:00,4515.90,4521.72,4486.54,4486.78,13827,58,0,NaN,NaN
997,2026-03-27 21:00:00,4486.76,4528.49,4486.63,4516.67,12654,58,0,NaN,NaN
998,2026-03-27 22:00:00,4516.58,4526.06,4500.64,4514.16,11856,58,0,NaN,NaN


In [17]:
trend_1h = df2['HighLow'].iloc[-1]
trend_1h

-1.0

In [18]:
smc.ob(df2, swing_highs_lows2)

,OB,Top,Bottom,OBVolume,MitigatedIndex,Percentage
0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
995,NaN,NaN,NaN,NaN,NaN,NaN
996,NaN,NaN,NaN,NaN,NaN,NaN
997,NaN,NaN,NaN,NaN,NaN,NaN
998,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
fvg2 = smc.fvg(df2)

In [24]:
# merge fvg df with df2 on index and keep all rows from df2
df2 = pd.merge(df2, fvg2, how='left', left_index=True, right_index=True)
df2

,time,open,high,low,close,volume,spread,real_volume,HighLow,level,FVG,Top,Bottom,MitigatedIndex
0,2026-01-27 11:00:00,5079.35,5099.46,5077.69,5098.71,10144,27,0,-1.0,5077.69,NaN,NaN,NaN,NaN
1,2026-01-27 12:00:00,5098.45,5098.45,5081.60,5086.24,9529,27,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-01-27 13:00:00,5086.25,5090.72,5067.01,5076.74,9246,27,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-01-27 14:00:00,5076.79,5090.97,5073.46,5084.95,11599,27,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-01-27 15:00:00,5084.86,5092.76,5069.61,5077.33,14447,27,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2026-03-27 19:00:00,4523.82,4533.54,4505.36,4515.94,13844,58,0,NaN,NaN,NaN,NaN,NaN,NaN
996,2026-03-27 20:00:00,4515.90,4521.72,4486.54,4486.78,13827,58,0,NaN,NaN,NaN,NaN,NaN,NaN
997,2026-03-27 21:00:00,4486.76,4528.49,4486.63,4516.67,12654,58,0,NaN,NaN,NaN,NaN,NaN,NaN
998,2026-03-27 22:00:00,4516.58,4526.06,4500.64,4514.16,11856,58,0,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# if trend in -1 sort the df2 with fvg column values of -1 and MitigatedIndex column values of 0 
# and return the first row of the sorted df2
if trend_1h == -1:
    df2_sorted = df2[(df2['FVG'] == -1) & (df2['MitigatedIndex'] == 0)].sort_values(by='time', ascending=False)
    print(df2_sorted.head(1))


                   time     open     high      low    close  volume  spread  \
877 2026-03-20 16:00:00  4678.54  4679.26  4612.61  4615.96   16762      55   

     real_volume  HighLow  level  FVG      Top   Bottom  MitigatedIndex  
877            0      NaN    NaN -1.0  4661.74  4622.21             0.0  


In [30]:
df2_sorted

,time,open,high,low,close,volume,spread,real_volume,HighLow,level,FVG,Top,Bottom,MitigatedIndex
877,2026-03-20 16:00:00,4678.54,4679.26,4612.61,4615.96,16762,55,0,NaN,NaN,-1.0,4661.74,4622.21,0.0
848,2026-03-19 10:00:00,4763.65,4784.08,4727.04,4727.66,12807,55,0,NaN,NaN,-1.0,4747.73,4740.12,0.0
847,2026-03-19 09:00:00,4827.32,4842.30,4747.73,4763.62,14130,55,0,NaN,NaN,-1.0,4826.80,4784.08,0.0
846,2026-03-19 08:00:00,4849.69,4853.62,4826.80,4827.29,8875,58,0,NaN,NaN,-1.0,4845.44,4842.30,0.0
829,2026-03-18 14:00:00,4967.68,4973.62,4924.89,4930.01,12957,55,0,NaN,NaN,-1.0,4958.78,4932.23,0.0
828,2026-03-18 13:00:00,4991.95,4991.96,4958.78,4967.66,8377,55,0,NaN,NaN,-1.0,4984.70,4973.62,0.0
763,2026-03-13 17:00:00,5101.54,5107.73,5035.66,5041.49,15576,55,0,NaN,NaN,-1.0,5096.43,5071.18,0.0
741,2026-03-12 18:00:00,5142.74,5144.64,5109.48,5132.17,14445,55,0,NaN,NaN,-1.0,5138.14,5137.85,0.0
740,2026-03-12 17:00:00,5150.93,5163.49,5138.14,5142.79,14818,55,0,NaN,NaN,-1.0,5145.99,5144.64,0.0
739,2026-03-12 16:00:00,5180.24,5185.87,5145.99,5151.07,14780,55,0,NaN,NaN,-1.0,5163.62,5163.49,0.0
